# 05 — Feature Engineering & Data Split

## Goal

This notebook exists to **test two things**:

1. **The created features** — that every column produced by the three feature modules is what it claims to be. Correct grain, correct derivation from the source columns, no missing values, no impossible values, and no column that silently encodes something other than its name.
2. **The train / validation / test split** — that the split is reproducible, that it partitions the customers cleanly, and that the three parts are comparable enough for a model tuned on one to be evaluated fairly on another.

Neither is exploratory. `04_eda_first_txn.ipynb` asked *what does the data say*; this notebook asks *is what we built correct*. Every cell here should be a check with a pass/fail reading, not a chart to interpret.

## The modules under test

Each one reads the same cleaned line items and produces a table at its own grain. The notebook calls their functions directly rather than reading their CSVs, so what is tested is the code that generates the features, not an artefact that may have gone stale on disk.

| Module | Produces | Grain | Rows |
|---|---|---|---|
| `src/build_features.py` | `customer_features.csv` | `customer_id` | 5,044 |
| `src/build_item_history.py` | `item_history.csv` | `(stock_code, date)` | 100,158 |
| `src/description_embeddings.py` | `description_embeddings.npz` | `stock_code` | 4,171 |

Only the first is a feature table. The other two are **lookups**: they sit at a grain other than the unit of prediction, so neither is a feature of a customer until it is aggregated to one — and how to aggregate it is a modelling decision left to the two pipelines. This notebook joins them onto the line items and stops there.

Each module ships a matching `check_*` function that asserts its way through the properties its table must satisfy. Those run here too, so a cell that executes without raising is itself the first pass/fail reading.

**Source:** `data/processed/first_transaction_churn_clean.csv` — the cleaned line-item table, one row per line item of each customer's first invoice, written by `src/correct_data_issues.py`. Everything below is built from it.

## What is being tested

### The features

`build_features.py` collapses 125,042 corrected line items into 5,044 customer rows across 16 columns. The item-history and embedding lookups are checked further down, where each is built. The checks below verify each group against the line-item source it came from, rather than trusting the aggregation:

| Group | Columns | What has to hold |
|---|---|---|
| Keys & label | `customer_id`, `churn` | one row per customer; label binary and unchanged from the source |
| Calendar parts | `year`, `month`, `day_of_month`, `weekday`, `hour`, `time_of_day` | each recomputable from `first_date`; buckets total and within domain |
| Category | `country`, `country_raw` | every named country above the frequency floor; pooled rows all `Other`; the raw column regroups back to the derived one |
| Roll-ups | `n_lines`, `n_distinct_products`, `total_quantity`, `total_spend`, `avg_unit_price` | each equal to the aggregation recomputed directly from the line items |

### The split

Whatever strategy is chosen, the split has to satisfy the same properties:

- **Complete and disjoint** — every customer lands in exactly one of train / validation / test; no customer appears twice.
- **Split on the customer** — the modelling grain is the customer, so the split unit must be `customer_id`. A row-level split would be meaningless here since there is already one row each, but the property is worth asserting so it stays true if the grain ever changes.
- **Reproducible** — the same seed produces the same partition on a re-run.
- **Proportioned as intended** — 70 / 15 / 15.
- **Comparable base rates** — the churn rate in each part is close enough that hyperparameters tuned on validation transfer to test. This is the property most at risk: `04` §6 shows churn moving between 31.2% and 72.4% month to month, so a time-ordered split does *not* satisfy it while a stratified one does.

### Open decisions this notebook depends on

Two choices are still unmade, and both change what the split cells should assert:

1. **Split strategy** — stratified random on `customer_id` (holds the base rate constant, isolates the pipeline comparison) versus time-ordered (realistic for deployment, but confounded by the drift above and needing a 90-day embargo between parts).
2. **The opening cohort** — the first 90 days hold 1,581 customers churning at 36.1% against 50.0% afterwards, most likely established customers whose earlier history predates the file. Keeping them, dropping them, or flagging them changes both the row count and the base rate the split has to preserve.

`src/split_data.py` does not exist yet.

In [1]:
# This notebook imports from src/, which is edited while the kernel is alive.
# Without autoreload, `from build_features import ...` is served from the module
# Python cached on first import, so a function or constant added to a .py file
# after the kernel started raises ImportError until the kernel is restarted.
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 180)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

RANDOM_SEED = 42

PROCESSED = Path('..') / 'data' / 'processed'
SRC = Path.cwd().parent / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# The starting point for everything below: one row per line item of each
# customer's first invoice, after the corrections in correct_data_issues.py.
lines = pd.read_csv(PROCESSED / 'first_transaction_churn_clean.csv',
                    parse_dates=['invoice_date'])

print(f'Cleaned first transactions : {lines.shape[0]:,} rows x {lines.shape[1]} columns')
print(f'Customers                  : {lines["customer_id"].nunique():,}')
print(f'Invoices                   : {lines["invoice"].nunique():,}')
print(f'Date range                 : {lines["invoice_date"].min():%Y-%m-%d} -> '
      f'{lines["invoice_date"].max():%Y-%m-%d}')
print(f'Churn rate                 : {lines.groupby("customer_id")["churn"].first().mean():.2%}')
print()
print(lines.dtypes.to_string())

lines.head(10)

Cleaned first transactions : 125,042 rows x 10 columns
Customers                  : 5,044
Invoices                   : 5,044
Date range                 : 2009-12-01 -> 2011-09-09
Churn rate                 : 45.64%

invoice                  int64
stock_code              object
description             object
quantity                 int64
invoice_date    datetime64[ns]
price                  float64
customer_id              int64
country                 object
line_total             float64
churn                    int64


,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country,line_total,churn
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.40,0
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.00,0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.00,0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.80,0
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.00,0
5,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085,United Kingdom,39.60,0
6,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.00,0
7,489434,21523,DOORMAT FANCY FONT HOME SWEET HOME,10,2009-12-01 07:45:00,5.95,13085,United Kingdom,59.50,0
8,489436,48173C,DOOR MAT BLACK FLOCK,10,2009-12-01 09:06:00,5.95,13078,United Kingdom,59.50,0
9,489436,21755,LOVE BUILDING BLOCK WORD,18,2009-12-01 09:06:00,5.45,13078,United Kingdom,98.10,0


## Building the variables

Two modules turn the table above into modelling inputs, split by grain. The cells below call them directly rather than reading their output:

| Table | Grain | Function |
|---|---|---|
| Customer modelling table | one row per `customer_id` | `build_features(lines)` from `src/build_features.py` |
| Item history lookup | one row per `(stock_code, date)` | `build_item_history(lines)` from `src/build_item_history.py` |

Each has a matching `check_*` function in the same module that asserts its way through the properties the table has to satisfy; both are run here, so a cell that executes without raising is itself the first pass/fail reading.

The item table is a **lookup**, not a feature table — which is why it lives in its own module and is deliberately not joined to the customer grain. How to summarise a basket's worth of product history to one row per customer is a modelling decision. Joining it onto the line items is therefore the next cell's job, and the aggregation to customer grain is left open.

### What gets added to the line items, and what does not

The dividing line is whether a value **varies within a basket**. Only what varies is computed here; everything constant is computed once per customer by `build_features.py` and joined after the collapse.

- **Joined on `(stock_code, date)`** — the six `prior_*` history columns, which differ product by product.
- **Derived per row** — the four line-vs-history ratios, which differ line by line.
- **Not here** — `year`, `month`, `weekday`, `hour`, `time_of_day`, `country`. These are identical on every line of a customer's single transaction, so deriving them at line grain would write the same value 18 times over (the median basket) and discard 17 of them. `build_features.py` derives them *after* grouping to one row per customer, which is also what keeps the country frequency floor counting customers rather than basket lines.
- **Not here either** — the customer-level roll-ups (`n_lines`, `total_quantity`, `total_spend`, `avg_unit_price`, …). Those are the *result* of aggregating these line items; broadcasting them back onto the rows they came from is how double counting starts.

`src/build_modelling_dataset.py` performs the collapse and the join.

In [2]:
from build_features import add_date_parts, build_features, check_modelling_table
from build_item_history import build_item_history, check_item_history

# Customer grain: one row per customer, the shared starting point for both pipelines.
features = build_features(lines)
check_modelling_table(features)

# (stock_code, date) grain: how each product had traded strictly before that date.
items = build_item_history(lines)
check_item_history(items, lines)

print(f'features : {features.shape[0]:,} rows x {features.shape[1]} columns  '
      f'(checks passed)')
print(f'items    : {items.shape[0]:,} rows x {items.shape[1]} columns  '
      f'(checks passed)')

# Both were built from `lines` here; the module also writes them to disk. Read the
# item lookup back to confirm the file and the function agree, then preview it.
items_file = pd.read_csv(PROCESSED / 'item_history.csv',
                         parse_dates=['date'])
assert items_file.shape == items.shape, 'item_history.csv is a different shape'
pd.testing.assert_frame_equal(items_file, items.reset_index(drop=True))
print('\nitem_history.csv matches build_item_history(lines)')

print(f'\nDistinct stock codes     : {items["stock_code"].nunique():,}')
print(f'Rows with no history yet : {(items["prior_transactions"] == 0).sum():,} '
      f'({(items["prior_transactions"] == 0).mean()*100:.1f}%) — every product\'s '
      f'first appearance')

items_file.head(10)

features : 5,044 rows x 16 columns  (checks passed)
items    : 100,158 rows x 8 columns  (checks passed)

item_history.csv matches build_item_history(lines)

Distinct stock codes     : 4,171
Rows with no history yet : 4,171 (4.2%) — every product's first appearance


,stock_code,date,prior_transactions,prior_units,prior_median_daily_transactions,prior_median_daily_units,prior_avg_price,prior_median_daily_price
0,10002,2009-12-01,0,0,NaN,NaN,NaN,NaN
1,10002,2009-12-03,1,12,1.00,12.00,0.85,0.85
2,10002,2009-12-04,4,19,2.00,9.50,0.85,0.85
3,10002,2009-12-06,8,92,3.00,12.00,0.85,0.85
4,10002,2009-12-07,9,140,2.00,30.00,0.85,0.85
5,10002,2009-12-11,10,142,1.00,12.00,0.85,0.85
6,10002,2009-12-14,11,151,1.00,10.50,0.85,0.85
7,10002,2010-01-04,13,187,1.00,12.00,0.85,0.85
8,10002,2010-01-11,14,190,1.00,10.50,0.85,0.85
9,10002,2010-01-14,15,238,1.00,12.00,0.85,0.85


In [3]:
items_file.head()

,stock_code,date,prior_transactions,prior_units,prior_median_daily_transactions,prior_median_daily_units,prior_avg_price,prior_median_daily_price
0,10002,2009-12-01,0,0,NaN,NaN,NaN,NaN
1,10002,2009-12-03,1,12,1.00,12.00,0.85,0.85
2,10002,2009-12-04,4,19,2.00,9.50,0.85,0.85
3,10002,2009-12-06,8,92,3.00,12.00,0.85,0.85
4,10002,2009-12-07,9,140,2.00,30.00,0.85,0.85


## Item history applied to the first transactions

Applying `src/build_features.py` to the cleaned first-transaction data, so every line item carries the trading history its product had **before** that transaction's date.

### How the metrics move over time

The module produces one row per `(stock_code, date)`, each computed from data strictly earlier than that date. A product bought on 3 March and again on 6 March therefore gets two different rows — the first sees everything up to 2 March, the second everything up to 5 March. The history grows as the product accumulates trade:

| `stock_code` | `date` | `prior_transactions` | `prior_units` |
|---|---|---|---|
| 85123A | 2009-12-01 | 0 | 0 |
| 85123A | 2009-12-03 | 31 | 1,003 |
| 85123A | 2009-12-06 | 61 | 1,415 |

Same-day transactions share a cutoff: because history is aggregated to whole days before any accumulation, all 18 customers who bought `85123A` on 2009-12-08 read the same 79 prior transactions, and none of them sees the other 17.

### The six history metrics

**Volume — how much the product had sold**

- **`prior_transactions`** — transactions containing this product before today.
- **`prior_units`** — units of it sold before today.
- **`prior_median_daily_transactions`** — on a typical earlier trading day, how many transactions included it.
- **`prior_median_daily_units`** — on a typical earlier trading day, how many units moved.

**Price level — what the product normally cost**

- **`prior_avg_price`** — mean, across earlier trading days, of that day's average unit price.
- **`prior_median_daily_price`** — median of the same daily prices, so a one-off promotion does not drag the level.

Both price columns are built from **daily** averages rather than from raw line items, matching how the unit metrics work. A day on which forty customers bought counts once, exactly as a one-customer day does — otherwise busy days would define "the usual price". This matters because `04` §8a found 1,681 codes (40.3%) selling at more than one price.

### Comparing the current line against that history

`add_history_ratios` adds four columns putting the line's own values next to what the product normally does:

- **`qty_vs_median_daily_units`** = `quantity / prior_median_daily_units`. Above 1 means this single line moved more than the product's whole typical day.
- **`qty_share_of_prior_units`** = `quantity / prior_units`. What fraction of everything ever sold of this product is being bought right now.
- **`price_vs_prior_avg_price`** = `price / prior_avg_price`. Above 1 means this customer paid a premium over the product's usual level; below 1 means they caught it discounted.
- **`price_vs_prior_median_daily_price`** = `price / prior_median_daily_price`. The same comparison against the median daily price instead of the mean.

The two price ratios differ only in how the historical level is summarised, and that difference is the point: a one-off promotion or a single odd trading day drags the mean but not the median. They agree for a stably-priced product and diverge for one whose history holds outlier days, so the gap between them is itself a signal about how erratically a product has been priced.

All four are `NaN` on a product's first-ever appearance, where there is no history to divide by. That is roughly 4% of rows and is a genuine undefined, not a missing value to impute.

In [4]:
# Item history onto every line item. (stock_code, date) is the lookup's key, so a
# many-to-one merge is the whole join: several line items may share a product-day.
txn = lines.assign(date=lines['invoice_date'].dt.normalize())
txn = txn.merge(items, on=['stock_code', 'date'], how='left', validate='many_to_one')

assert len(txn) == len(lines), 'the merge changed the row count'
assert txn['prior_transactions'].notna().all(), 'a line item found no history row'

# Nothing constant within a customer is derived here — no calendar parts, no
# country grouping. Those live in `features`, one row per customer already, and
# are joined after the collapse. Deriving them on 125,042 line items would repeat
# each value once per basket line only to discard all but one of them.
assert (txn.groupby('customer_id')['prior_transactions'].size()
        == features.set_index('customer_id')['n_lines']).all(), \
    'line counts no longer agree with n_lines'

added = [c for c in txn.columns if c not in lines.columns]
print(f'Enriched line items : {txn.shape[0]:,} rows x {txn.shape[1]} columns')
print(f'Added {len(added)} columns: {", ".join(added)}')
print()
print('Every added column varies within a basket, so every one of them has to be '
      'aggregated\nto reach the customer grain. The constant ones come from '
      '`features` at the join.')

txn.head(5)

Enriched line items : 125,042 rows x 17 columns
Added 7 columns: date, prior_transactions, prior_units, prior_median_daily_transactions, prior_median_daily_units, prior_avg_price, prior_median_daily_price

Every added column varies within a basket, so every one of them has to be aggregated
to reach the customer grain. The constant ones come from `features` at the join.


,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country,line_total,churn,date,prior_transactions,prior_units,prior_median_daily_transactions,prior_median_daily_units,prior_avg_price,prior_median_daily_price
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.40,0,2009-12-01,0,0,NaN,NaN,NaN,NaN
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.00,0,2009-12-01,0,0,NaN,NaN,NaN,NaN
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.00,0,2009-12-01,0,0,NaN,NaN,NaN,NaN
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.80,0,2009-12-01,0,0,NaN,NaN,NaN,NaN
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.00,0,2009-12-01,0,0,NaN,NaN,NaN,NaN


In [5]:
from build_item_history import RATIO_COLUMNS, add_history_ratios

# Line-vs-history ratios. Denominators are guarded inside the function: prior_units
# is 0 on a product's first appearance and the medians and means are NaN there, so
# a debut row comes out NaN rather than infinite.
txn = add_history_ratios(txn)

print(f'Line items with history joined : {len(txn):,}')
print(f'Ratio columns added            : {", ".join(RATIO_COLUMNS)}')
print(f'Undefined on a debut row       : '
      f'{txn["qty_share_of_prior_units"].isna().sum():,} '
      f'({txn["qty_share_of_prior_units"].isna().mean()*100:.1f}%)')

# The two price ratios summarise the same history differently, so where they
# disagree the product's price history holds outlier days.
both = txn[RATIO_COLUMNS[2:]].dropna()
gap = (both['price_vs_prior_avg_price'] - both['price_vs_prior_median_daily_price']).abs()
print(f'\nMean vs median price ratio     : identical to 3dp on '
      f'{(gap < 5e-4).mean()*100:.1f}% of rows, '
      f'largest gap {gap.max():.2f}')
print(both.describe(percentiles=[0.5, 0.99]).round(3).to_string())

Line items with history joined : 125,042
Ratio columns added            : qty_vs_median_daily_units, qty_share_of_prior_units, price_vs_prior_avg_price, price_vs_prior_median_daily_price
Undefined on a debut row       : 5,659 (4.5%)

Mean vs median price ratio     : identical to 3dp on 51.7% of rows, largest gap 3.32
       price_vs_prior_avg_price  price_vs_prior_median_daily_price
count                119,383.00                         119,383.00
mean                       1.00                               0.99
std                        0.15                               0.16
min                        0.03                               0.03
50%                        1.00                               1.00
99%                        1.18                               1.11
max                       26.25                              26.25


In [6]:
ITEM_HISTORY_COLUMNS = [
    'date',
    'customer_id',
    'stock_code',
    'quantity',
    'price',
    'prior_transactions',
    'prior_units',
    'prior_median_daily_transactions',
    'prior_median_daily_units',
    'prior_avg_price',
    'prior_median_daily_price',
    *RATIO_COLUMNS,
]

item_history = txn[ITEM_HISTORY_COLUMNS].sort_values(['date', 'customer_id', 'stock_code'])
print(f'{len(item_history):,} rows x {item_history.shape[1]} columns')

# The opening day is every product's first appearance, so it is all NaN by
# construction and makes a poor preview. Show a mid-period transaction instead.
mid = item_history[item_history['prior_transactions'] > 0]
example_customer = mid.iloc[len(mid) // 2]['customer_id']

print(f'\nOne complete transaction — customer {example_customer}:')
item_history[item_history['customer_id'] == example_customer].head(5)

125,042 rows x 15 columns

One complete transaction — customer 16894:


,date,customer_id,stock_code,quantity,price,prior_transactions,prior_units,prior_median_daily_transactions,prior_median_daily_units,prior_avg_price,prior_median_daily_price,qty_vs_median_daily_units,qty_share_of_prior_units,price_vs_prior_avg_price,price_vs_prior_median_daily_price
64371,2010-06-22,16894,15036,12,0.75,57,2827,1.00,24.00,0.70,0.75,0.50,0.00,1.08,1.00
64229,2010-06-22,16894,15044A,1,2.95,23,110,1.00,6.00,2.95,2.95,0.17,0.01,1.00,1.00
64228,2010-06-22,16894,15044B,1,2.95,18,124,1.00,6.00,2.93,2.95,0.17,0.01,1.01,1.00
64227,2010-06-22,16894,15044C,1,2.95,18,75,1.00,3.00,2.95,2.95,0.33,0.01,1.00,1.00
64212,2010-06-22,16894,15056BL,2,5.95,77,1196,1.00,3.00,5.85,5.95,0.67,0.00,1.02,1.00


In [7]:

# mid['customer_id'].unique()
mid[mid['customer_id'] == 12437]
# mid

,date,customer_id,stock_code,quantity,price,prior_transactions,prior_units,prior_median_daily_transactions,prior_median_daily_units,prior_avg_price,prior_median_daily_price,qty_vs_median_daily_units,qty_share_of_prior_units,price_vs_prior_avg_price,price_vs_prior_median_daily_price
3670,2009-12-02,12437,20724,10,0.85,5,110,5.00,110.00,0.85,0.85,0.09,0.09,1.00,1.00
3672,2009-12-02,12437,20749,4,7.95,2,4,2.00,4.00,7.95,7.95,1.00,1.00,1.00,1.00
3673,2009-12-02,12437,20750,4,7.95,4,29,4.00,29.00,7.15,7.15,0.14,0.14,1.11,1.11
3680,2009-12-02,12437,20977,16,1.25,1,4,1.00,4.00,1.25,1.25,4.00,4.00,1.00,1.00
3681,2009-12-02,12437,20979,16,1.25,3,36,3.00,36.00,1.25,1.25,0.44,0.44,1.00,1.00
3683,2009-12-02,12437,20981,12,0.85,1,1,1.00,1.00,0.85,0.85,12.00,12.00,1.00,1.00
3682,2009-12-02,12437,20983,12,0.85,3,26,3.00,26.00,0.85,0.85,0.46,0.46,1.00,1.00
3669,2009-12-02,12437,21429,16,1.65,2,12,2.00,12.00,1.65,1.65,1.33,1.33,1.00,1.00
3674,2009-12-02,12437,21432,8,5.95,1,8,1.00,8.00,5.95,5.95,1.00,1.00,1.00,1.00
3685,2009-12-02,12437,21754,6,5.95,6,51,6.00,51.00,5.87,5.87,0.12,0.12,1.01,1.01


## Description embeddings

`src/description_embeddings.py` encodes each product's `description` with SBERT (`all-MiniLM-L6-v2`, 384 dimensions), so text that carries real meaning — `RED HANGING HEART T-LIGHT HOLDER` — stops being an opaque string.

### Grain, again

Descriptions belong to the **product**, not the line item. `correct_data_issues.normalize_descriptions` has already collapsed every `stock_code` to one modal description, so the 125,042 line items hold only 4,171 distinct `(stock_code, description)` pairs. Embedding the line items would encode the same strings thirty times over; the module embeds the catalogue once and leaves the join to the caller, exactly as the item history does.

So this is a third table, at a third grain:

| Table | Grain | Rows |
|---|---|---|
| `features` | `customer_id` | 5,044 |
| `items` | `(stock_code, date)` | 100,158 |
| `products` / `product_embeddings` | `stock_code` | 4,171 |

Turning a 384-dimensional product vector into a feature of a *customer* means choosing how to pool a basket's worth of vectors — mean, max, tf-idf weighting, cluster assignment, distance to the customer's own centroid. That is a modelling decision, so it is left to the two pipelines rather than done here.

### Why the dot product is the similarity

`embed_descriptions` returns L2-normalised rows, so `a @ b` is exactly the cosine similarity of `a` and `b`, bounded in [-1, 1]. On unnormalised vectors a plain dot product would also reward magnitude, which for sentence embeddings tracks description length more than meaning. Normalising once up front means a whole similarity matrix is one matrix multiply.

In [8]:
from description_embeddings import (MATMUL_ERRSTATE, build_description_embeddings,
                                    check_embeddings, load_model, most_similar)

# One model instance, reused by the next cell too.
sbert = load_model()

# Product grain: 125,042 line items -> one embedding per distinct stock_code.
products, product_embeddings = build_description_embeddings(lines, model=sbert)
check_embeddings(products, product_embeddings)

print(f'Products embedded  : {len(products):,} (from {len(lines):,} line items)')
print(f'Embedding matrix   : {product_embeddings.shape} {product_embeddings.dtype}  '
      f'({product_embeddings.nbytes / 1e6:.1f} MB in memory)')
print(f'Row norms          : all 1.0 '
      f'(max deviation {abs(np.linalg.norm(product_embeddings, axis=1) - 1).max():.2e})')

# Distinct codes can share a description — colour and size variants whose
# distinguishing detail never reached the text. They embed identically, so a
# similarity of exactly 1.0 between two different codes is expected below.
shared = products.groupby('description')['stock_code'].size()
shared = shared[shared > 1]
print(f'Shared descriptions: {len(shared)} descriptions cover {int(shared.sum())} products, '
      f'e.g. {shared.idxmax()!r} x{shared.max()}')

# products.iloc[i] describes product_embeddings[i] — check_embeddings asserts it.
preview = products.head(10).copy()
for d in range(4):
    preview[f'dim_{d}'] = product_embeddings[:10, d]
preview

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Products embedded  : 4,171 (from 125,042 line items)
Embedding matrix   : (4171, 384) float32  (6.4 MB in memory)
Row norms          : all 1.0 (max deviation 1.19e-07)
Shared descriptions: 27 descriptions cover 59 products, e.g. 'COLUMBIAN CANDLE ROUND' x4


,stock_code,description,dim_0,dim_1,dim_2,dim_3
0,10002,INFLATABLE POLITICAL GLOBE,0.08,0.03,0.00,-0.01
1,10080,GROOVY CACTUS INFLATABLE,0.03,0.04,-0.07,0.04
2,10109,BENDY COLOUR PENCILS,-0.10,-0.05,0.02,-0.01
3,10120,DOGGY RUBBER,-0.04,-0.02,0.04,0.03
4,10123C,HEARTS WRAPPING TAPE,-0.05,0.06,0.03,-0.00
5,10123G,ARMY CAMO WRAPPING TAPE,-0.10,0.03,-0.04,-0.02
6,10124A,SPOTS ON RED BOOKCOVER TAPE,-0.04,-0.05,-0.06,0.01
7,10124G,ARMY CAMO BOOKCOVER TAPE,-0.09,-0.01,-0.09,-0.06
8,10125,MINI FUNKY DESIGN TAPES,-0.06,0.03,0.00,-0.08
9,10133,COLOURING PENCILS BROWN TUBE,0.00,-0.04,-0.03,-0.07


In [9]:
# A sample description, taken from the catalogue so its own row appears at
# similarity 1.0 — the end-to-end proof that the matrix rows are still aligned
# with `products`.
QUERY_STOCK_CODE = '85123A'
TOP_N = 10

position = products.index[products['stock_code'] == QUERY_STOCK_CODE][0]
query = products.loc[position, 'description']
query_vec = product_embeddings[position]

# The dot product against every product. Both sides are unit length, so this
# *is* the cosine similarity — no scaling, no normalising, no special function.
# errstate silences spurious Accelerate BLAS flags; see MATMUL_ERRSTATE.
with np.errstate(**MATMUL_ERRSTATE):
    similarity = product_embeddings @ query_vec

order = np.argsort(-similarity)[:TOP_N]
neighbours = (products.iloc[order]
              .assign(similarity=similarity[order])
              .reset_index(drop=True))

print(f'Query: {QUERY_STOCK_CODE}  {query!r}')
print(f'Similarity range over all {len(products):,} products: '
      f'{similarity.min():.3f} to {similarity.max():.3f}, mean {similarity.mean():.3f}')

# The module helper must agree. Compared as sets and sorted values rather than
# row-for-row: identical descriptions give identical similarities, and the two
# code paths (argsort here, argpartition there) may break those ties differently.
helper = most_similar(query, products, product_embeddings, model=sbert, top_n=TOP_N)
assert set(helper['stock_code']) == set(neighbours['stock_code']), 'helper disagrees'
assert np.allclose(sorted(helper['similarity']), sorted(neighbours['similarity'])), \
    'helper similarities differ'
print('most_similar() returns the same neighbours as the dot product above')

# 4 decimals here rather than the notebook-wide 2, so the exact ties are visible.
FMT = '{:.4f}'.format
ties = neighbours['similarity'].duplicated(keep=False)
if ties.any():
    print(f'\n{ties.sum()} of the top {TOP_N} tie exactly — products sharing one description:')
    print(neighbours[ties].to_string(index=False, float_format=FMT))

with pd.option_context('display.float_format', FMT):
    display(neighbours)

Query: 85123A  'WHITE HANGING HEART T-LIGHT HOLDER'
Similarity range over all 4,171 products: -0.069 to 1.000, mean 0.244
most_similar() returns the same neighbours as the dot product above

2 of the top 10 tie exactly — products sharing one description:
stock_code          description  similarity
     21814 HEART T-LIGHT HOLDER      0.8554
     85118 HEART T-LIGHT HOLDER      0.8554


,stock_code,description,similarity
0,85123A,WHITE HANGING HEART T-LIGHT HOLDER,1.0000
1,21733,RED HANGING HEART T-LIGHT HOLDER,0.9013
2,21814,HEART T-LIGHT HOLDER,0.8554
3,85118,HEART T-LIGHT HOLDER,0.8554
4,84970S,HANGING HEART ZINC T-LIGHT HOLDER,0.8356
5,35968,FOLK ART METAL HEART T-LIGHT HOLDER,0.8299
6,84978,HANGING HEART JAR T-LIGHT HOLDER,0.8213
7,21313,GLASS HEART T-LIGHT HOLDER,0.7997
8,84949,SILVER HANGING T-LIGHT HOLDER,0.7752
9,22789,T-LIGHT HOLDER SWEETHEART HANGING,0.7583


### Attaching the embeddings to the line items

The whole 384-dimensional vector goes into **one** column rather than 384 float columns, so the frame stays readable and the vector stays a single object to aggregate.

Every line item of a given product points at the *same* array: the catalogue has 4,171 products behind 125,042 line items, so a per-row copy would inflate 6.4 MB into 192 MB without adding any information. Those shared arrays are marked read-only, since writing through one row would otherwise silently rewrite every other row of the same product.

To reach the customer grain later, stack and average:

```python
customer_vectors = (txn.groupby('customer_id')[EMBEDDING_COLUMN]
                       .apply(lambda s: stack_embeddings(s).mean(axis=0)))
```

One thing to decide when you do: the stored vectors are unit length, but **a mean of unit vectors is not**. Across these customers the mean vector's norm runs from 0.42 to 1.00, median 0.54 — it reaches 1.0 only for a basket whose products all share one description, and falls as the basket gets more varied. That norm is a real signal about basket coherence, not an artefact, so re-normalise only if you want direction without it.

In [10]:
from description_embeddings import (EMBEDDING_COLUMN, EMBEDDING_DIM,
                                    attach_embeddings, pca_column_names,
                                    stack_embeddings)

txn = attach_embeddings(txn, products, product_embeddings)
vectors = txn[EMBEDDING_COLUMN]

print(f'Enriched line items : {txn.shape[0]:,} rows x {txn.shape[1]} columns')
print(f'New column          : {EMBEDDING_COLUMN} — one '
      f'{vectors.iloc[0].shape[0]}-dim {vectors.iloc[0].dtype} vector per row')
print(f'Distinct arrays     : {len({id(v) for v in vectors}):,} for {len(txn):,} rows '
      f'(one per product, shared and read-only)')
print(f'Column memory       : {vectors.memory_usage(deep=True) / 1e6:.1f} MB, against '
      f'{len(txn) * EMBEDDING_DIM * 4 / 1e6:.0f} MB if each row held its own copy')

# Every row must carry its own product's catalogue vector, not a neighbour's.
positions = products.set_index('stock_code').index.get_indexer(txn['stock_code'])
assert np.array_equal(stack_embeddings(vectors), product_embeddings[positions]), \
    'a line item is holding the wrong product vector'
print('Every row matches its stock_code\'s catalogue vector')

# --------------------------------------------------------------------------
# The finished line-item table: item history and embeddings side by side
# --------------------------------------------------------------------------
# A subset of the 28 columns — enough to see that each line carries both what
# its product had done before that date and what its description means.
SHOWCASE_COLUMNS = [
    'customer_id', 'stock_code', 'description', 'date', 'quantity', 'price',
    'prior_transactions', 'prior_units', 'prior_median_daily_units',
    'prior_avg_price', 'price_vs_prior_avg_price',
    'price_vs_prior_median_daily_price', EMBEDDING_COLUMN,
]

# One complete transaction, from a customer whose products all have history —
# a product's debut day is NaN by construction and makes a poor preview.
with_history = txn[txn['prior_transactions'] > 0]
example_customer = with_history.iloc[len(with_history) // 2]['customer_id']
showcase = txn[txn['customer_id'] == example_customer][SHOWCASE_COLUMNS].copy()

# Abbreviate the vectors: pandas renders an object cell with str(), which would
# otherwise print all 384 values or clip them mid-number.
showcase[EMBEDDING_COLUMN] = [np.array2string(v, precision=2, threshold=6,
                                              edgeitems=2, suppress_small=True)
                              for v in showcase[EMBEDDING_COLUMN]]

print(f'\nCustomer {example_customer} — {len(showcase)} line items, '
      f'{len(SHOWCASE_COLUMNS)} of {txn.shape[1]} columns shown')
print('  item history  -> prior_* and price_vs_prior_avg_price, from build_item_history.py')
print(f'  embeddings    -> {EMBEDDING_COLUMN}, from description_embeddings.py')

with pd.option_context('display.width', 250, 'display.max_colwidth', 40):
    display(showcase.head(12))

Enriched line items : 125,042 rows x 22 columns
New column          : description_embedding — one 384-dim float32 vector per row
Distinct arrays     : 4,171 for 125,042 rows (one per product, shared and read-only)
Column memory       : 15.0 MB, against 192 MB if each row held its own copy


Every row matches its stock_code's catalogue vector

Customer 14793 — 25 line items, 13 of 22 columns shown
  item history  -> prior_* and price_vs_prior_avg_price, from build_item_history.py
  embeddings    -> description_embedding, from description_embeddings.py


,customer_id,stock_code,description,date,quantity,price,prior_transactions,prior_units,prior_median_daily_units,prior_avg_price,price_vs_prior_avg_price,price_vs_prior_median_daily_price,description_embedding
64412,14793,82597,BOMBS AWAY METAL SIGN,2010-06-22,12,2.10,38,214,4.00,2.10,1.00,1.00,[-0.02 0.1 ... -0.01 0.03]
64413,14793,22357,KINGS CHOICE BISCUIT TIN,2010-06-22,4,4.25,57,189,4.00,4.24,1.00,1.00,[-0.05 0.03 ... -0. -0.04]
64414,14793,22139,RETRO SPOT TEA SET CERAMIC 11 PC,2010-06-22,3,4.95,188,881,6.00,4.92,1.01,1.00,[-0.03 -0. ... -0.07 -0.03]
64415,14793,21790,VINTAGE SNAP CARDS,2010-06-22,12,0.85,145,2050,12.00,0.84,1.01,1.00,[-0.07 0.09 ... -0.07 0.08]
64416,14793,21888,BINGO SET,2010-06-22,4,3.75,81,447,4.00,3.73,1.01,1.00,[-0.04 0.05 ... 0.01 0.07]
64417,14793,22621,TRADITIONAL KNITTING NANCY,2010-06-22,12,1.45,24,323,15.00,1.43,1.01,1.00,[-0.03 0.01 ... -0.03 0.02]
64418,14793,22045,SPACEBOY GIFT WRAP,2010-06-22,25,0.42,37,1125,25.00,0.42,1.00,1.00,[-0.13 0.1 ... -0.07 0.04]
64419,14793,22046,TEA PARTY WRAPPING PAPER,2010-06-22,25,0.42,21,575,25.00,0.42,1.00,1.00,[-0.11 0.03 ... 0.05 0.04]
64420,14793,22047,EMPIRE GIFT WRAP,2010-06-22,25,0.42,50,1700,25.00,0.42,1.01,1.00,[-0.12 0.07 ... -0.01 0.08]
64421,14793,22497,SET OF 2 TINS VINTAGE BATHROOM,2010-06-22,4,4.25,15,142,4.00,4.22,1.01,1.00,[ 0.02 0.03 ... -0.08 -0.01]


In [11]:
txn[txn['invoice'] == 489669]

,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country,line_total,churn,date,prior_transactions,prior_units,prior_median_daily_transactions,prior_median_daily_units,prior_avg_price,prior_median_daily_price,qty_vs_median_daily_units,qty_share_of_prior_units,price_vs_prior_avg_price,price_vs_prior_median_daily_price,description_embedding
1958,489669,21733,RED HANGING HEART T-LIGHT HOLDER,24,2009-12-02 08:15:00,2.95,12842,United Kingdom,70.80,1,2009-12-02,5,80,5.00,80.00,2.87,2.87,0.30,0.30,1.03,1.03,"[-0.0006294052, 0.07581325, 0.0021737705, 0.05..."
1959,489669,84970S,HANGING HEART ZINC T-LIGHT HOLDER,24,2009-12-02 08:15:00,0.85,12842,United Kingdom,20.40,1,2009-12-02,2,24,2.00,24.00,0.85,0.85,1.00,1.00,1.00,1.00,"[-0.047372043, 0.14260651, 0.016657941, 0.0371..."
1960,489669,22086,PAPER CHAIN KIT 50'S CHRISTMAS,120,2009-12-02 08:15:00,2.55,12842,United Kingdom,306.00,1,2009-12-02,18,343,18.00,343.00,2.91,2.91,0.35,0.35,0.88,0.88,"[-0.121155255, 0.10488178, 0.020022094, -0.009..."


## Assembling the modelling dataset

`src/build_modelling_dataset.py` performs the collapse: line items in, one row per customer out. It composes the three feature modules rather than deriving anything new, in four stages.

| Stage | Function | Result |
|---|---|---|
| 1 | `build_line_metrics(lines)` | line grain, carrying the 6 `prior_*` and the 4 ratios |
| 2 | `aggregate_line_metrics(txn)` | 5,044 x 50 — each metric by min/max/mean/median/sum |
| 3 | `aggregate_embeddings(txn, ...)` | 5,044 x 384 — the basket's mean SBERT vector |
| 4 | join `customer_features` | 5,044 x 450 |

The cells below run each stage separately and check it against a hand recomputation, then confirm the module's own `build_modelling_dataset` reproduces the same table end to end. Each stage is checked for the thing that could actually go wrong with it:

- **Stage 1** — that only genuinely line-varying columns are carried forward. A column that is constant within a basket has no business being aggregated.
- **Stage 2** — that the five aggregations are what they claim, including the `NaN` handling for a customer whose whole basket lacks history.
- **Stage 3** — that the mean is over the customer's *line items*, so a product bought twice counts twice.
- **Stage 4** — that the customer-level columns arrive **unchanged**, not silently reordered or recomputed.

In [12]:
from build_modelling_dataset import (AGGREGATE_COLUMNS, AGGREGATIONS, EMBEDDING_COLUMNS,
                                     LINE_METRIC_COLUMNS, aggregate_embeddings,
                                     aggregate_line_metrics, build_line_metrics,
                                     build_modelling_dataset, check_modelling_dataset,
                                     reduce_customer_embeddings)

# ---- Stage 1: the line-varying metrics ------------------------------------
line_metrics = build_line_metrics(lines)

# The module must reproduce exactly what this notebook built by hand in the
# item-history cells above — same rows, same values, for every metric column.
carried = ['date'] + LINE_METRIC_COLUMNS
pd.testing.assert_frame_equal(line_metrics[carried], txn[carried])
print(f'build_line_metrics reproduces the notebook\'s txn on all '
      f'{len(carried)} carried columns')

# Every aggregated column must actually vary within a basket, or aggregating it
# is busywork hiding a constant.
varies = {c: txn.groupby('customer_id')[c].nunique(dropna=False).max() > 1
          for c in LINE_METRIC_COLUMNS}
assert all(varies.values()), f'constant column being aggregated: {[c for c, v in varies.items() if not v]}'
print(f'All {len(LINE_METRIC_COLUMNS)} metric columns vary within a basket')

# And nothing constant per customer sneaked in.
constant = [c for c in line_metrics.columns
            if c not in ('customer_id',)
            and line_metrics.groupby('customer_id')[c].nunique(dropna=False).max() <= 1]
print(f'Constant-per-customer columns present : {constant}')
print('  (invoice, country, churn and date are inputs carried from the source,')
print('   not aggregated — LINE_METRIC_COLUMNS is what stage 2 consumes)')

build_line_metrics reproduces the notebook's txn on all 11 carried columns
All 10 metric columns vary within a basket
Constant-per-customer columns present : ['invoice', 'country', 'churn', 'date']
  (invoice, country, churn and date are inputs carried from the source,
   not aggregated — LINE_METRIC_COLUMNS is what stage 2 consumes)


In [13]:
# ---- Stage 2: aggregation ---------------------------------------------------
aggregates = aggregate_line_metrics(line_metrics)
print(f'aggregates : {aggregates.shape[0]:,} rows x {aggregates.shape[1]} columns '
      f'({len(LINE_METRIC_COLUMNS)} metrics x {len(AGGREGATIONS)} aggregations)')
assert list(aggregates.columns) == AGGREGATE_COLUMNS, 'unexpected aggregate columns'
assert len(aggregates) == lines['customer_id'].nunique(), 'not one row per customer'

# Recompute every one of the 50 by hand for a single customer.
CHECK_CUSTOMER = 14793
basket = line_metrics[line_metrics['customer_id'] == CHECK_CUSTOMER]

by_hand = {}
for col in LINE_METRIC_COLUMNS:
    values = basket[col]
    for agg in AGGREGATIONS:
        by_hand[f'{col}_{agg}'] = (values.sum(min_count=1) if agg == 'sum'
                                   else getattr(values, agg)())
by_hand = pd.Series(by_hand)[AGGREGATE_COLUMNS]

mismatch = (by_hand - aggregates.loc[CHECK_CUSTOMER]).abs().max()
assert mismatch < 1e-9, f'aggregation mismatch: {mismatch}'
print(f'\nCustomer {CHECK_CUSTOMER} ({len(basket)} line items): all '
      f'{len(AGGREGATE_COLUMNS)} aggregates match a hand recomputation')

# Shown for two metrics, so the five aggregations are visible as numbers.
shown = pd.DataFrame(
    {agg: [by_hand[f'{c}_{agg}'] for c in ('prior_units', 'price_vs_prior_avg_price')]
     for agg in AGGREGATIONS},
    index=['prior_units', 'price_vs_prior_avg_price'])
display(shown.round(3))

aggregates : 5,044 rows x 50 columns (10 metrics x 5 aggregations)

Customer 14793 (25 line items): all 50 aggregates match a hand recomputation


,min,max,mean,median,sum
prior_units,20.00,"2,827.00",639.92,447.00,"15,998.00"
price_vs_prior_avg_price,1.00,1.08,1.01,1.00,25.17


In [14]:
# ---- Stage 2b: the customers whose whole basket has no history --------------
# Every product in these baskets is making its first-ever appearance, so all four
# ratios are NaN on every line. pandas sums an all-NaN group to 0.0, which would
# hand them a fabricated zero while min/max/mean/median correctly read NaN — and 0
# is a meaningful value for a ratio, so the fabrication would be invisible.
blank = line_metrics.groupby('customer_id')['qty_share_of_prior_units'].apply(
    lambda s: s.isna().all())
blank = blank[blank].index
print(f'Customers with no usable history : {len(blank):,}')

ratio_aggregates = [f'qty_share_of_prior_units_{a}' for a in AGGREGATIONS]
blank_rows = aggregates.loc[blank, ratio_aggregates]
assert blank_rows.isna().all().all(), 'an aggregation invented a value for a blank basket'
print(f'All five aggregations are NaN for every one of them (min_count=1 on the sum)')

# The naive version, for contrast — this is what the sum would have been.
naive = line_metrics.groupby('customer_id')['qty_share_of_prior_units'].sum()
print(f'\nWithout min_count=1, sum would report '
      f'{naive.loc[blank].iloc[0]:.1f} for these customers instead of NaN')

# The prior_* columns behave differently and should: a total with no history is a
# genuine 0 (nothing had sold yet), not an undefined.
print(f'prior_units_sum for the same customers : '
      f'{aggregates.loc[blank, "prior_units_sum"].unique()}  <- 0, not NaN, by design')

Customers with no usable history : 92
All five aggregations are NaN for every one of them (min_count=1 on the sum)

Without min_count=1, sum would report 0.0 for these customers instead of NaN
prior_units_sum for the same customers : [0]  <- 0, not NaN, by design


In [15]:
# ---- Stage 3a: average the product vectors per customer ---------------------
embedding_means = aggregate_embeddings(line_metrics, products, product_embeddings)
print(f'embedding_means : {embedding_means.shape[0]:,} rows x '
      f'{embedding_means.shape[1]} columns')
assert list(embedding_means.columns) == EMBEDDING_COLUMNS, 'unexpected embedding columns'
assert np.isfinite(embedding_means.to_numpy()).all(), 'a mean is NaN or Inf'

# By hand for the same customer: index the catalogue matrix to this basket's rows
# and average. Averaging over LINE ITEMS, so a product on two lines counts twice.
positions = pd.Index(products['stock_code']).get_indexer(basket['stock_code'])
by_hand_vector = product_embeddings[positions].mean(axis=0)
from_module = embedding_means.loc[CHECK_CUSTOMER].to_numpy()

print(f'\nCustomer {CHECK_CUSTOMER}: max |by hand - aggregate_embeddings| = '
      f'{np.abs(by_hand_vector - from_module).max():.2e}')
assert np.abs(by_hand_vector - from_module).max() < 1e-6, 'embedding mean mismatch'

# A mean of unit vectors is not unit length, and the shortfall is real signal:
# it falls as a basket gets more varied.
norms = np.linalg.norm(embedding_means.to_numpy(), axis=1)
print(f'Mean-vector norms : min {norms.min():.3f}, median {np.median(norms):.3f}, '
      f'max {norms.max():.3f}  (1.0 only for a single-description basket)')


# ---- Stage 3b: reduce the customer means with PCA ---------------------------
# These 384 means are intermediate — only the reduced dimensions reach the
# dataset. PCA is fitted here, on the customer vectors, rather than on the
# product catalogue: averaging ~18 unit vectors per basket smooths them, so 90%
# of the variance survives in 96 components where the catalogue needs 140.
embedding_components, pca = reduce_customer_embeddings(embedding_means)

print(f'\nPCA : {embedding_means.shape[1]} means -> {pca.n_components_} components '
      f'({pca.explained_variance_ratio_.sum():.1%} of variance)')
assert list(embedding_components.columns) == pca_column_names(pca.n_components_), \
    'reduced dimensions are misnamed'
assert embedding_components.index.equals(embedding_means.index), 'customer order changed'

# The reduction must be exactly the projection of the means it was fitted on.
assert np.abs(pca.transform(embedding_means.to_numpy())
              - embedding_components.to_numpy()).max() < 1e-5, 'not the PCA projection'
print('Reduced values are the PCA projection of the means, customer order preserved')

cumulative = np.cumsum(pca.explained_variance_ratio_)
print(f'Variance by component : 1st {cumulative[0]:.1%}, 10th {cumulative[9]:.1%}, '
      f'25th {cumulative[24]:.1%}, {pca.n_components_}th {cumulative[-1]:.1%}')

# PCA is affine, so it commutes with averaging — reducing the products first and
# averaging after gives the same numbers, given the same components. What the
# order decides is which matrix the components are fitted on.
print(f'\nDropped {embedding_means.shape[1] - pca.n_components_} dimensions '
      f'for {1 - cumulative[-1]:.1%} of the variance')

embedding_means : 5,044 rows x 384 columns

Customer 14793: max |by hand - aggregate_embeddings| = 1.11e-08
Mean-vector norms : min 0.417, median 0.536, max 1.000  (1.0 only for a single-description basket)



PCA : 384 means -> 96 components (90.1% of variance)
Reduced values are the PCA projection of the means, customer order preserved
Variance by component : 1st 8.7%, 10th 39.5%, 25th 60.5%, 96th 90.1%

Dropped 288 dimensions for 9.9% of the variance


/opt/anaconda3/lib/python3.13/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: divide by zero encountered in matmul
  X_transformed = X @ self.components_.T
/opt/anaconda3/lib/python3.13/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: overflow encountered in matmul
  X_transformed = X @ self.components_.T
/opt/anaconda3/lib/python3.13/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: invalid value encountered in matmul
  X_transformed = X @ self.components_.T
/opt/anaconda3/lib/python3.13/site-packages/sklearn/decomposition/_base.py:155: RuntimeWarning: divide by zero encountered in matmul
  X_transformed -= xp.reshape(self.mean_, (1, -1)) @ self.components_.T
/opt/anaconda3/lib/python3.13/site-packages/sklearn/decomposition/_base.py:155: RuntimeWarning: overflow encountered in matmul
  X_transformed -= xp.reshape(self.mean_, (1, -1)) @ self.components_.T
/opt/anaconda3/lib/python3.13/site-packages/sklearn/decomposition/_base.py:155: R

In [16]:
# ---- Stage 4: join the customer-level columns -------------------------------
dataset_piecewise = (features
                     .merge(aggregates, on='customer_id', how='left', validate='one_to_one')
                     .merge(embedding_components, on='customer_id', how='left',
                            validate='one_to_one'))

# The customer-grain columns must arrive untouched — same values, same order, same
# dtypes. This is the check that the join did not reorder or recompute anything.
pd.testing.assert_frame_equal(dataset_piecewise[features.columns], features)
print(f'All {features.shape[1]} customer-level columns joined unchanged '
      f'(assert_frame_equal against build_features output)')

assert len(dataset_piecewise) == len(features), 'the join changed the row count'
assert not dataset_piecewise['customer_id'].duplicated().any(), 'customer_id is not unique'
print(f'Grain preserved : {len(dataset_piecewise):,} rows, one per customer')

# ---- The module end to end --------------------------------------------------
dataset = build_modelling_dataset(lines, model=sbert)
check_modelling_dataset(dataset, lines, model=sbert)
pd.testing.assert_frame_equal(dataset, dataset_piecewise)
print('\nbuild_modelling_dataset reproduces the four stages exactly, '
      'and its own checks pass')

print(f'\nModelling dataset : {dataset.shape[0]:,} rows x {dataset.shape[1]} columns')
print(f'  {features.shape[1]:>3} customer-level  (churn, calendar, country, roll-ups)')
print(f'  {len(AGGREGATE_COLUMNS):>3} aggregates      ({len(LINE_METRIC_COLUMNS)} metrics x {len(AGGREGATIONS)})')
print(f'  {pca.n_components_:>3} PCA dimensions  (from {len(EMBEDDING_COLUMNS)} means, '
      f'{cumulative[-1]:.0%} of variance)')

All 16 customer-level columns joined unchanged (assert_frame_equal against build_features output)
Grain preserved : 5,044 rows, one per customer



build_modelling_dataset reproduces the four stages exactly, and its own checks pass

Modelling dataset : 5,044 rows x 162 columns
   16 customer-level  (churn, calendar, country, roll-ups)
   50 aggregates      (10 metrics x 5)
   96 PCA dimensions  (from 384 means, 90% of variance)


In [17]:
# The finished table: one row per customer, a slice of each column group.
SLICE = (['customer_id', 'churn', 'first_date', 'weekday', 'time_of_day', 'country',
          'n_lines', 'total_spend']
         + [f'prior_units_{a}' for a in ('min', 'max', 'mean')]
         + ['qty_share_of_prior_units_sum', 'price_vs_prior_avg_price_median']
         + list(embedding_components.columns[:3]))

print(f'{len(SLICE)} of {dataset.shape[1]} columns shown\n')
with pd.option_context('display.width', 260, 'display.max_colwidth', 22):
    display(dataset[SLICE].head(8).round(3))

16 of 162 columns shown



,customer_id,churn,first_date,weekday,time_of_day,country,n_lines,total_spend,prior_units_min,prior_units_max,prior_units_mean,qty_share_of_prior_units_sum,price_vs_prior_avg_price_median,pca_00,pca_01,pca_02
0,12347,0,2010-10-31 14:20:00,6,afternoon,Other,40,611.53,14,5337,677.30,3.43,1.00,0.05,-0.00,-0.04
1,12348,0,2010-09-27 14:59:00,0,afternoon,Other,19,221.16,27,7859,"2,225.68",1.77,1.00,0.07,0.00,-0.28
2,12350,1,2011-02-02 16:01:00,2,afternoon,Other,16,294.40,12,1556,534.81,1.49,1.00,0.01,0.03,0.13
3,12351,1,2010-11-29 15:23:00,0,afternoon,Other,21,300.93,0,2000,469.67,12.74,1.00,0.00,-0.06,0.03
4,12352,0,2010-11-12 10:20:00,4,morning,Other,6,143.75,135,1720,950.67,0.11,1.00,-0.03,-0.02,0.12
5,12353,1,2010-10-27 12:44:00,2,afternoon,Other,20,317.76,20,4654,706.00,1.07,1.00,0.04,-0.11,-0.02
6,12354,1,2011-04-21 13:11:00,3,afternoon,Spain,58,"1,079.40",0,3175,583.45,3.03,1.00,0.12,-0.01,0.05
7,12355,1,2010-05-21 11:59:00,4,morning,Other,22,488.21,3,1035,156.54,28.43,1.00,0.07,-0.01,0.04


In [18]:
dataset.head()

,customer_id,churn,first_date,year,month,day_of_month,weekday,hour,time_of_day,country,country_raw,n_lines,n_distinct_products,total_quantity,total_spend,avg_unit_price,prior_transactions_min,prior_transactions_max,prior_transactions_mean,prior_transactions_median,prior_transactions_sum,prior_units_min,prior_units_max,prior_units_mean,prior_units_median,prior_units_sum,prior_median_daily_transactions_min,prior_median_daily_transactions_max,prior_median_daily_transactions_mean,prior_median_daily_transactions_median,prior_median_daily_transactions_sum,prior_median_daily_units_min,prior_median_daily_units_max,prior_median_daily_units_mean,prior_median_daily_units_median,prior_median_daily_units_sum,prior_avg_price_min,prior_avg_price_max,prior_avg_price_mean,prior_avg_price_median,prior_avg_price_sum,prior_median_daily_price_min,prior_median_daily_price_max,prior_median_daily_price_mean,prior_median_daily_price_median,prior_median_daily_price_sum,qty_vs_median_daily_units_min,qty_vs_median_daily_units_max,qty_vs_median_daily_units_mean,qty_vs_median_daily_units_median,...,pca_46,pca_47,pca_48,pca_49,pca_50,pca_51,pca_52,pca_53,pca_54,pca_55,pca_56,pca_57,pca_58,pca_59,pca_60,pca_61,pca_62,pca_63,pca_64,pca_65,pca_66,pca_67,pca_68,pca_69,pca_70,pca_71,pca_72,pca_73,pca_74,pca_75,pca_76,pca_77,pca_78,pca_79,pca_80,pca_81,pca_82,pca_83,pca_84,pca_85,pca_86,pca_87,pca_88,pca_89,pca_90,pca_91,pca_92,pca_93,pca_94,pca_95
0,12347,0,2010-10-31 14:20:00,2010,10,31,6,14,afternoon,Other,Iceland,40,40,509,611.53,1.83,3,242,48.17,30.00,1927,14,5337,677.30,327.50,27092,1.00,1.00,1.00,1.00,40.00,2.00,48.00,12.38,10.00,495.00,0.40,12.68,1.81,1.25,72.29,0.38,12.75,1.81,1.25,72.31,0.25,8.00,1.37,1.00,...,-0.01,0.01,0.01,-0.00,-0.00,0.02,0.00,-0.01,0.02,-0.01,0.02,0.00,0.01,0.03,-0.02,-0.01,0.01,-0.01,0.00,-0.00,-0.00,0.00,0.00,0.00,0.02,0.02,0.01,-0.02,0.02,-0.00,-0.01,0.02,-0.01,0.00,0.01,0.00,0.00,-0.01,-0.00,0.01,-0.02,0.00,0.00,0.01,-0.02,0.02,0.01,0.00,0.02,-0.00
1,12348,0,2010-09-27 14:59:00,2010,9,27,0,14,afternoon,Other,Finland,19,19,372,221.16,0.70,3,213,65.58,39.00,1246,27,7859,"2,225.68",916.00,42288,1.00,2.00,1.11,1.00,21.00,4.00,48.00,24.45,24.00,464.50,0.29,1.45,0.70,0.55,13.38,0.29,1.45,0.70,0.55,13.39,0.50,3.00,1.13,1.00,...,-0.03,0.02,0.01,-0.02,0.02,-0.01,-0.02,0.00,0.01,-0.00,-0.02,-0.01,-0.03,-0.02,0.01,0.00,-0.04,-0.02,-0.00,0.03,0.03,0.04,-0.01,0.00,-0.01,0.00,-0.01,0.01,0.02,-0.00,0.00,0.02,0.01,-0.02,-0.01,0.00,0.02,-0.01,0.01,-0.02,-0.02,0.00,0.00,0.01,-0.01,-0.00,-0.01,0.02,0.02,0.00
2,12350,1,2011-02-02 16:01:00,2011,2,2,2,16,afternoon,Other,Norway,16,16,196,294.40,1.58,5,143,59.69,56.50,955,12,1556,534.81,319.50,8557,1.00,1.00,1.00,1.00,16.00,1.00,12.00,6.38,5.25,102.00,0.85,2.95,1.58,1.55,25.28,0.85,2.95,1.58,1.55,25.30,1.00,6.00,2.90,2.29,...,-0.01,0.02,-0.00,0.01,-0.02,-0.03,-0.00,0.01,-0.00,-0.00,-0.02,0.01,-0.02,0.02,0.03,-0.01,-0.01,-0.00,-0.03,0.02,-0.02,-0.00,-0.00,-0.02,-0.01,0.00,-0.00,-0.02,0.01,-0.03,0.00,-0.01,-0.00,-0.00,-0.02,0.02,-0.02,0.02,0.01,0.01,0.01,0.03,0.01,0.02,-0.01,-0.02,-0.02,-0.03,-0.00,-0.01
3,12351,1,2010-11-29 15:23:00,2010,11,29,0,15,afternoon,Other,Unspecified,21,21,261,300.93,2.36,0,95,38.00,26.00,798,0,2000,469.67,314.00,9863,1.00,1.00,1.00,1.00,20.00,1.00,25.00,11.55,6.00,231.00,0.41,12.69,2.38,1.82,47.58,0.42,12.75,2.39,1.82,47.81,0.50,12.00,1.65,1.00,...,0.00,0.01,0.02,-0.02,-0.00,0.02,-0.04,-0.01,-0.01,-0.01,-0.00,-0.05,0.02,0.01,-0.00,-0.00,0.02,-0.03,0.03,0.03,-0.03,-0.01,0.03,-0.01,-0.00,-0.02,-0.02,0.03,-0.00,-0.00,-0.01,-0.01,0.04,-0.01,-0.01,0.01,0.01,-0.00,-0.00,0.01,0.02,0.01,-0.01,-0.01,-0.01,0.01,0.01,-0.00,-0.01,0.00
4,12352,0,2010-11-12 10:20:00,2010,11,12,4,10,morning,Other,Norway,6,6,77,143.75,2.07,24,151,97.67,104.50,586,135,1720,950.67,"1,019.00",5704,1.00,1.00,1.00,1.00,6.00,3.50,24.00,8.58,5.50,51.50,1.25,2.95,2.07,2.08,12.40,1.25,2.95,2.07,2.10,12.45,1.00,3.43,1.89,1.75,...,-0.00,-0.02,-0.07,0.03,0.01,-0.02,0.02,-0.02,-0.00,-0.04,0.03,0.01,-0.04,0.00,-0.01,-0.04